# Design Problem ANN solutions


## 1 Problem definition

After thorough inspection of the contents in disc-benchmark-files, we have deduced the problem to two subproblems (deduced from example scripts), that closely relate to each other, and the materials discussed during the first 3 lectures. The main problem being "1-step-ahead prediction of the next state given past k input and output sequences.

As shown in the two example scripts (and discussed in the practical excercises) we will first train our model on predicting the next state using the provided dataset at each prediction set. After that we will validate our models performance by using each models observation predictions as input for its next prediction.


## 1.1 Models
This notewook encapsulates the ANN part of the M4SC Design Project and thus contains two simpler models and a more advanced deep neural network.

# 3. Advanced Neural network


## 3.1 Approach

As defined in practical excercise set 3, Problem 2. A common approach for choosing neural network structures to solve a problem is by:

1. Looking up what others have used on similar problems and using that as a starting point.
2. Conducting a hyperparameter search using cross-validation around that starting point.

Hence rather than emperically testing a bunch of networks we have conducted research. Nonlinear system identification based on NARX network (IEEE) Treats NARX as an Recurrent neural network and shows it is able to "outperform standard neural networks such as FTDNN and Elman architectures". Neural network-based parametric system identification: a limitation we found in a source however adresses that "One limitation of the typical recurrent layer is that the time window of its inputs and outputs is required to be fixed during the training.". But as the training data contains a fixed window of 15 steps we will not have to overcome this limitation. However it has been noted in practical excercise set 3 that Regular RNN's structure to capture long term dependencies effectively. 

The following cells will cover: 

- Data preparation
-  Model Definition
- Model Training
- Simulation evaluation

## 3.2 Data preparation

The exmaple prediction/simulation data shows 15th order lookback on input and outputs. Hence we will need to prepare the data such that our input shape becomes:
(N, 15, 2): N times: 15 samples for input u and output y.

In [44]:
import torch
from torch import nn
from torch.nn import functional as F
import pandas as df
import numpy as np

data = np.loadtxt("../gym-unbalanced-disk/disc-benchmark-files/training-val-test-data.csv", delimiter=",", skiprows=1)
window_size = 15

# Prepare the data for training Window sized observations and the corresponding target values (1 step future output)
def prepare_windows(data, window_size):
    X = []
    y = []
    for i in range(window_size, len(data)):
        X.append(data[i-window_size:i, :])  # All columns except the last one
        y.append(data[i, -1])  # The last column is the target
    return np.array(X), np.array(y)

X, y = prepare_windows(data, window_size)

# Split the data into training and test: sets 80% for training and 20% for testing
train_size = 0.8
X_train, X_test =  X[:int(0.8 * len(X))], X[int(0.8 * len(X)):]
y_train, y_test = y[:int(0.8 * len(y))], y[int(0.8 * len(y)):]


# Create a validation set from the training data 10% of the training data will be used for validation
validation_size = 0.1
X_val, y_val = X_train[-int(validation_size * len(X_train)):], y_train[-int(validation_size * len(y_train)):]
X_train, y_train = X_train[:-int(validation_size * len(X_train))], y_train[:-int(validation_size * len(y_train))]

# Normalize the data 
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# First we scale the training features and then we use the same scaler to transform the validation and test features. 
# This prevents data leakage.
X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_val = scaler.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape)
X_test = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)  

# For target value we need to scale using just the th mean and std of the training target values
th_mean = scaler.mean_[-1]
th_std = scaler.scale_[-1]

y_train = (y_train - th_mean) / th_std
y_val = (y_val - th_mean) / th_std
y_test = (y_test - th_mean) / th_std

# Convert the data to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

## 3.3 Model definition

Here we shall perform model definition

In [45]:
class simple_lstm(nn.Module):
    def __init__(self, hidden_size, input_shape, num_layers=1, h2o_nodes=40):
        super(simple_lstm, self).__init__()
        self.hidden_size = hidden_size
        self.input_size = input_shape[1]
        self.output_size = 1

        net = lambda n_in,n_out: nn.Sequential(nn.Linear(n_in,h2o_nodes),nn.Sigmoid(),nn.Linear(h2o_nodes,n_out))

        self.lstm = nn.LSTM(input_size=self.input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True).float()
        self.h2o = net(hidden_size + self.input_size, self.output_size).float()

    def forward(self, inputs):
        hiddens, _ = self.lstm(inputs)
        combined = torch.cat((hiddens, inputs), dim=2)
        h2o_input = combined.view(-1, self.hidden_size + self.input_size)
        y_predict = self.h2o(h2o_input).view(inputs.shape[0], inputs.shape[1])
        return y_predict[:, -1]

In [ ]:
import optuna
def objective(trial):
    hidden_size = trial.suggest_int('hidden_size', 10, 100)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    num_layers = trial.suggest_int('num_layers', 1, 5)
    h2o_nodes = trial.suggest_int('h2o_nodes', 10, 100)

    model = simple_lstm(hidden_size, X_train.shape[1:], num_layers=num_layers, h2o_nodes=h2o_nodes)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    num_epochs = 50
    batch_size = 64
    best_val = float('inf')
    patience, no_improve = 10, 0

    for epoch in range(num_epochs):
        model.train()
        for i in range(0, len(X_train), batch_size):
            X_batch = X_train[i:i+batch_size]
            y_batch = y_train[i:i+batch_size]
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val), y_val).item()
        if val_loss < best_val:
            best_val = val_loss
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    return best_val

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)
print("Best hyperparameters: ", study.best_params)

[I 2026-05-20 15:24:08,224] A new study created in memory with name: no-name-baefda98-b8f4-4e4c-87ee-22c51cdd9579
[I 2026-05-20 15:24:40,568] Trial 0 finished with value: 0.0007804666529409587 and parameters: {'hidden_size': 97, 'learning_rate': 0.0003330218178923491, 'num_layers': 2, 'h2o_nodes': 18}. Best is trial 0 with value: 0.0007804666529409587.
[I 2026-05-20 15:25:47,042] Trial 1 finished with value: 0.0005375482724048197 and parameters: {'hidden_size': 38, 'learning_rate': 0.0001897242236075093, 'num_layers': 4, 'h2o_nodes': 39}. Best is trial 1 with value: 0.0005375482724048197.
[I 2026-05-20 15:27:08,870] Trial 2 finished with value: 7.297859701793641e-05 and parameters: {'hidden_size': 70, 'learning_rate': 0.0008182854194124317, 'num_layers': 3, 'h2o_nodes': 32}. Best is trial 2 with value: 7.297859701793641e-05.
[I 2026-05-20 15:27:24,289] Trial 3 finished with value: 0.00011691131658153608 and parameters: {'hidden_size': 37, 'learning_rate': 0.0013389037250395711, 'num_la